# Deep Learning A Z Convolucional - MNIST augmentation

In [1]:
!pip install tensorflow==2.16.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.8/589.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 37.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 39.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 29.4 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.2.0
    Uninstalling ml-dtypes-0.2.0:
      Successfully uninstalled ml-dtypes-0.2.0
  Attempting uninstall: h5py
    Found existing installation: h5py 3.9.0
    Uninstalling h5py-3.9.0:
      Successfully uninstalled h5py-3.9.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.15.2
    Uninstalling tensorboard-2.15.2:
      Successfully uninstalled tensorboard-2.15.2
  Attempting uninstall: keras
    Fo

In [2]:
import keras
import tensorflow as tf

In [3]:
keras.__version__, tf.__version__

('3.4.1', '2.16.1')

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras import utils as np_utils

In [ ]:
# Carrega os dados de treinamento
(X_treinamento, y_treinamento), (X_teste, y_teste) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Faz o reshape dos dados para o formato que a rede neural espera
X_treinamento = X_treinamento.reshape(X_treinamento.shape[0], 28, 28, 1)
X_teste = X_teste.reshape(X_teste.shape[0], 28, 28, 1)
# Faz a normalizacao dos dados para um intervalo entre 0 e 1
X_treinamento = X_treinamento.astype('float32')
X_teste = X_teste.astype('float32')
X_treinamento /= 255
X_teste /= 255
# Faz a codificacao das classes para o formato one-hot encoding
# Como a classe um ficaria [0, 1, 0, 0, 0, 0, 0, 0, 0, 0] a classe dois ficaria [0, 0, 1, 0, 0, 0, 0, 0, 0, 0] e assim por diante
y_treinamento = np_utils.to_categorical(y_treinamento, 10)
y_teste = np_utils.to_categorical(y_teste, 10)

In [ ]:
# Cria a rede neural convolucional
classificador = Sequential()
# Adiciona a camada de entrada com o formato da img de 28x28 com um canal de cor, ou seja, preto e branco
classificador.add(InputLayer(shape=(28, 28, 1)))
# Adiciona a camada convolucional com 32 filtros de tamanho 3x3 e a funcao de ativacao ReLU
classificador.add(Conv2D(32, (3, 3), activation='relu'))
# Adiciona a camada de pooling com o tamanho da janela de 2x2
# O pooling vai servir para reduzir a dimensionalidade dos dados, basicamente ele vai pegar a janela de 2x2
# e vai pegar o valor maximo dessa janela
classificador.add(MaxPooling2D(pool_size=(2, 2)))
# Adiciona a camada de flatten para transformar os dados em um vetor
classificador.add(Flatten())
# Adiciona a camada densa com 128 neuronios e a funcao de ativacao ReLU
classificador.add(Dense(units=128, activation='relu'))
# Adiciona a camada de saida com 10 neuronios, uma para cada classe
# a funcao de ativacao softmax vai servir para transformar os valores de saida em probabilidades
classificador.add(Dense(units=10, activation='softmax'))
# Compila a rede neural com a funcao de perda categorical_crossentropy, o otimizador Adam e a metrica de acuracia
classificador.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Cria um gerador de dados para aumentar o conjunto de dados de treinamento
# e o augmentation, ele vai criar novas imagens a partir das imagens de treinamento
# ele aplica alteracoes nas imagens de treinamento como rotacao, flip horizontal, etc 
gerador_treinamento = ImageDataGenerator(rotation_range=7, horizontal_flip=True,
                                         shear_range=0.2, height_shift_range=0.07,
                                         zoom_range=0.2)

In [ ]:
# Cria um gerador de dados para o conjunto de dados de teste
# nesse caso nao vai aplicar o augmentation vai so normalizar os dados
gerador_teste = ImageDataGenerator()

In [ ]:
# Gerador de dados para o conjunto de dados de treinamento com o augmentation e batch size de 128
base_treinamento = gerador_treinamento.flow(X_treinamento, y_treinamento, batch_size = 128)

In [ ]:
# Gerador de dados para o conjunto de dados de teste sem o augmentation e batch size de 128
base_teste = gerador_teste.flow(X_teste, y_teste, batch_size = 128)

In [ ]:
# Traina a rede neural com o gerador de dados de treinamento e usa o gerador de dados de teste para validacao
classificador.fit(base_treinamento, epochs=5, validation_data=base_teste)

Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


469/469 ━━━━━━━━━━━━━━━━━━━━ 58s 119ms/step - accuracy: 0.7557 - loss: 0.7821 - val_accuracy: 0.9423 - val_loss: 0.1852
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 82s 118ms/step - accuracy: 0.9252 - loss: 0.2438 - val_accuracy: 0.9632 - val_loss: 0.1166
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 80s 115ms/step - accuracy: 0.9469 - loss: 0.1754 - val_accuracy: 0.9685 - val_loss: 0.0981
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 54s 113ms/step - accuracy: 0.9553 - loss: 0.1401 - val_accuracy: 0.9702 - val_loss: 0.0888
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 83s 116ms/step - accuracy: 0.9585 - loss: 0.1300 - val_accuracy: 0.9743 - val_loss: 0.0757
